In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
Pressure = inflation.InflatableSheet.EnergyType.Pressure
Elastic  = inflation.InflatableSheet.EnergyType.Elastic
Full     = inflation.InflatableSheet.EnergyType.Full

In [ ]:
n_vx = [[0, 0], [0, 1], [0, 2],
        [1, 0], [1, 1], [1, 2],
        [2, 0], [2, 1], [2, 2]]
n_edge = [(0, 1), (1, 2), 
          (3, 4), (4, 5),
          (6, 7), (7, 8),
          (0, 3), (3, 6),
          (1, 4), (4, 7),
          (2, 5), (5, 8)]
triArea = 0.5

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")

In [ ]:
fuseMarkers = [0] * 9

In [ ]:
fuseMarkers[4] = 1

In [ ]:
fuseMarkers[0] = 1

In [ ]:
fuseMarkers[1] = 1
fuseMarkers[2] = 1

fuseMarkers[6] = 1
fuseMarkers[7] = 1
fuseMarkers[8] = 1

In [ ]:
fuseMarkers

In [ ]:
np.array(fuseMarkers) == 1

In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=5, height=5)

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) == 1)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) == 1)

In [ ]:
isheet.pressure = 10

In [ ]:
# perturb = np.random.random(isheet.numVars()) * 1e-3

In [ ]:
perturb = np.random.random(isheet.numVars()) * 1e-3

In [ ]:
isheet.setVars(isheet.getVars() + perturb)

In [ ]:
ipu.setVars(ipu.getVars() + np.random.random(ipu.numVars()) * 1e-1)

In [ ]:
ipu.getVars()

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
np.set_printoptions(precision = 5, suppress = True)

In [ ]:
import fd_validation

In [ ]:
ipu.sheet.pressure = 100

In [ ]:
ipu.get_use_planar_homogenization()

In [ ]:
fd_validation.gradConvergencePlot(isheet, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Pressure})

In [ ]:
fd_validation.hessConvergencePlot(isheet, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Elastic})

In [ ]:
fd_validation.gradConvergencePlot(isheet, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Elastic})

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"energyType": Pressure})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": Elastic})

In [ ]:
from periodic_simulation_setup import *

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = 3

In [ ]:
import fd_validation

In [ ]:
fixedVars, hessianShift = [], 1e-6

In [ ]:
benchmark.reset()

opts.niter = 10
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
benchmark.report()